[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C63_ML_System_Design_Course/02_data_design/02_data_system_design.ipynb)

# 02 · 数据系统设计（成本-质量-合规画像 / 标注体系 / 泄漏防范 / 长尾 / 隐私 / 血缘 / 冷启动 / 存储吞吐）

目标：把「数据系统怎么设计」从一段口头描述，变成几个**可以运行、可以断言**的小工具。

本 notebook 你会亲手实现：
1. **标注成本模型与预算分配** —— 从单价推总预算，再在预算有限时决定优先给哪个类别补标注
2. **泄漏检测器** —— 构造 track / 地理 / 设备 / 时间四类泄漏，并各自验证「坏切分触发、干净切分不触发」
3. **数据规模需求反推** —— 用两个试点观测点拟合学习曲线，反推达到目标指标需要多少样本
4. **分层抽样与代表性检验** —— 用卡方统计量 + 蒙特卡洛 p 值，纯 numpy 判断一份样本代表不代表总体
5. **存储与吞吐量级计算器** —— 车队原始数据量、训练读取吞吐，各自的数量级
6. **冷启动策略决策树** —— 给定先验条件，代码化地选出该走哪条路

> 心智模型：**这一模块讲的都是「在建模开始之前，必须想清楚的事」——数据系统设计得好不好，直接决定了模型能力的天花板。**

## 1 · 标注成本模型与预算分配

成本 = 基础单价 × 数量 × 复杂度系数 + 返工成本 + 质检固定开销。预算有限时，按「缺口从大到小」贪心分配。

In [ ]:
import numpy as np, math, random
from collections import Counter

def annotation_cost(n, p0=0.5, c_complexity=2.5, rework_rate=0.08, rework_multiplier=1.2, qc_fixed=0.0):
    """标注总成本：基础成本 + 返工成本 + 质检固定开销。
    p0: 单帧基础单价；c_complexity: 任务复杂度乘数（多类别检测通常 2-4 倍于二分类）；
    rework_rate: 质检发现的返工比例；rework_multiplier: 返工的相对代价系数。"""
    base = n * p0 * c_complexity
    rework = n * rework_rate * p0 * c_complexity * rework_multiplier
    return base + rework + qc_fixed

total = annotation_cost(1_000_000, qc_fixed=50_000)
print(f'100 万帧、多类别检测复杂度系数 2.5、返工率 8%：总预算 ≈ {total:,.0f} 元')
assert abs(total - 1_420_000) < 1e-6
print('其中返工成本占比：', f"{(total - 1_000_000*0.5*2.5 - 50_000) / total:.1%}")
print('\n✅ 成本模型就位：面试里能把「基础成本 / 返工成本 / 质检开销」三项分别报数量级，比只报一个总数更有说服力。')

In [ ]:
def allocate_budget(gaps, unit_cost, budget):
    """gaps: {类别: 需要补标注的样本数缺口}。按缺口从大到小贪心分配预算，
    直到预算用尽或缺口全部补齐。返回 (实际分配, 剩余预算)。"""
    order = sorted(gaps, key=lambda k: -gaps[k])
    alloc = {k: 0 for k in gaps}
    remaining = budget
    for k in order:
        need = gaps[k]
        cost_needed = need * unit_cost
        if cost_needed <= remaining:
            alloc[k] = need
            remaining -= cost_needed
        else:
            alloc[k] = int(remaining // unit_cost)
            remaining -= alloc[k] * unit_cost
            break
    return alloc, remaining

gaps = {'限速5': 500, '限速120': 300, '停车让行': 1000}   # 三个长尾类别的补标注缺口
alloc, remaining = allocate_budget(gaps, unit_cost=1.25, budget=2000)
print('分配结果：', alloc, '| 剩余预算：', remaining)
assert alloc == {'限速5': 500, '限速120': 100, '停车让行': 1000}
assert remaining == 0
print('\n✅ 预算优先给缺口最大的「停车让行」，「限速120」只补到预算耗尽为止——')
print('   这正是长尾类别补标注时最常见的真实约束：预算不够覆盖所有缺口，必须显式排优先级。')

## 2 · 泄漏检测器：构造并检出四类泄漏

track / 地理 / 设备 / 时间，各自构造一个「随机切分（坏）」与一个「按分组切分（干净）」，
验证检测器能在坏切分上报警、在干净切分上保持沉默。

In [ ]:
def group_overlap(train_ids, test_ids):
    """分组键在训练集与测试集之间的交集——非空即泄漏。"""
    return set(np.unique(train_ids)) & set(np.unique(test_ids))

def has_time_leak(train_time, test_time):
    """时间泄漏：训练集里存在比测试集最早样本还晚的时间点，说明切分不是严格「早训练、晚测试」。"""
    return bool(train_time.max() > test_time.min())

# ── 场景 A：track 级泄漏（同一段连续视频的帧被随机拆到两边）──
rng_a = np.random.default_rng(1)
n_tracks, frames_per_track = 20, 10
track_id = np.repeat(np.arange(n_tracks), frames_per_track)     # 20 条 track，每条 10 帧
idx = np.arange(len(track_id))

bad_idx = rng_a.permutation(idx)                                 # 坏切分：忽略 track 边界随机打散
split = int(0.8 * len(idx))
bad_train, bad_test = bad_idx[:split], bad_idx[split:]
leak_bad_track = group_overlap(track_id[bad_train], track_id[bad_test])
assert len(leak_bad_track) > 0, '随机切分理应产生 track 泄漏'

clean_test_tracks = set(range(16, 20))                            # 干净切分：整段 track 归一边
clean_mask = np.isin(track_id, list(clean_test_tracks))
leak_clean_track = group_overlap(track_id[idx[~clean_mask]], track_id[idx[clean_mask]])
assert len(leak_clean_track) == 0, '按 track 分组切分不应有重叠'

print(f'track 泄漏 —— 坏切分重叠 {len(leak_bad_track)} 条 track；干净切分重叠 {len(leak_clean_track)} 条 track')

In [ ]:
# ── 场景 B：地理泄漏（同一路口/路段同时出现在两边）──
rng_b = np.random.default_rng(2)
n = 500
location_id = rng_b.integers(0, 8, size=n)                        # 8 个地理位置
idx = np.arange(n)
bad_idx = rng_b.permutation(idx)
split = int(0.8 * n)
bad_train, bad_test = bad_idx[:split], bad_idx[split:]
leak_bad_geo = group_overlap(location_id[bad_train], location_id[bad_test])
assert len(leak_bad_geo) > 0

clean_mask = np.isin(location_id, [6, 7])                         # 干净切分：整片地点只留给测试集
leak_clean_geo = group_overlap(location_id[idx[~clean_mask]], location_id[idx[clean_mask]])
assert len(leak_clean_geo) == 0
print(f'地理泄漏 —— 坏切分重叠 {len(leak_bad_geo)} 个地点；干净切分重叠 {len(leak_clean_geo)} 个地点')

# ── 场景 C：设备泄漏（同一相机/标定同时出现在两边）──
rng_c = np.random.default_rng(3)
device_id = rng_c.integers(0, 3, size=n)                          # 3 种设备
bad_idx = rng_c.permutation(idx)
bad_train, bad_test = bad_idx[:split], bad_idx[split:]
leak_bad_dev = group_overlap(device_id[bad_train], device_id[bad_test])
assert len(leak_bad_dev) > 0

clean_mask = device_id == 2                                       # 干净切分：设备 2 整体留给测试集
leak_clean_dev = group_overlap(device_id[idx[~clean_mask]], device_id[idx[clean_mask]])
assert len(leak_clean_dev) == 0
print(f'设备泄漏 —— 坏切分重叠 {len(leak_bad_dev)} 种设备；干净切分重叠 {len(leak_clean_dev)} 种设备')

# ── 场景 D：时间泄漏（切分不满足「训练集时间早于测试集」）──
rng_d = np.random.default_rng(4)
n2 = 1000
timestamp = np.arange(n2)                                         # 严格递增的时间戳
idx2 = np.arange(n2)
bad_idx = rng_d.permutation(idx2)
split2 = int(0.8 * n2)
bt, bte = bad_idx[:split2], bad_idx[split2:]
leak_bad_time = has_time_leak(timestamp[bt], timestamp[bte])
assert leak_bad_time is True

ct, cte = idx2[:split2], idx2[split2:]                            # 干净切分：滚动切分，训练集严格更早
leak_clean_time = has_time_leak(timestamp[ct], timestamp[cte])
assert leak_clean_time is False
print(f'时间泄漏 —— 坏切分: {leak_bad_time}；干净切分: {leak_clean_time}')

print('\n✅ 四类泄漏全部「坏切分触发、干净切分不触发」——这正是「随机划分在感知数据里几乎总是错的」的可执行证据。')

## 3 · 数据规模需求反推：从学习曲线拟合到样本量

用一个饱和型学习曲线 $p(n) = 1 - a \cdot n^{-b}$ 近似「样本量 → 指标」的关系，
两个试点观测点就能拟合 $a, b$，再反解「达到目标指标需要多少样本」。

In [ ]:
def learning_curve(n, a=5.0, b=0.4):
    """饱和型学习曲线：n 越大指标越接近 1，但边际收益递减。"""
    return 1.0 - a * (np.asarray(n, dtype=float) ** -b)

def required_n_closed_form(target, a, b):
    """闭式反解：target = 1 - a n^-b  =>  n = (a / (1-target)) ** (1/b)。"""
    assert 0 < target < 1
    return (a / (1 - target)) ** (1 / b)

def required_n_binary_search(target, a, b, n_max=10**12):
    """二分反解，作为闭式解的独立校验（learning_curve 对 n 单调递增，二分合法）。"""
    lo, hi = 1.0, float(n_max)
    for _ in range(200):
        mid = (lo + hi) / 2
        if learning_curve(mid, a, b) >= target:
            hi = mid
        else:
            lo = mid
    return hi

a0, b0 = 5.0, 0.4
n90_cf = required_n_closed_form(0.9, a0, b0)
n90_bs = required_n_binary_search(0.9, a0, b0)
print(f'闭式解: {n90_cf:,.0f}   二分解: {n90_bs:,.0f}')
assert abs(n90_cf - n90_bs) / n90_cf < 1e-6
assert learning_curve(n90_cf, a0, b0) >= 0.9 - 1e-9

print('\n✅ 闭式解与二分解一致：反推样本量不需要靠猜，只要有一个单调的「样本量→指标」模型就能解。')

## 4 · 分层抽样与代表性检验

分层抽样：按各层在总体里的真实占比抽样。代表性检验：用卡方统计量 + 蒙特卡洛模拟出的 p 值判断
「这份样本的分布，像不像是从总体里按真实比例抽出来的」——全程纯 numpy，不依赖 scipy。

In [ ]:
def stratified_sample(population, strata_values, n_total, rng):
    """按各层在总体中的占比等比例抽样；名额向下取整后的剩余名额补给样本量最大的层。"""
    counts = {s: int(np.sum(population == s)) for s in strata_values}
    total = len(population)
    alloc = {s: int(n_total * counts[s] / total) for s in strata_values}
    remainder = n_total - sum(alloc.values())
    if remainder > 0:
        biggest = max(strata_values, key=lambda s: counts[s])
        alloc[biggest] += remainder
    sample_idx = []
    for s in strata_values:
        idx_s = np.where(population == s)[0]
        sample_idx.append(rng.choice(idx_s, size=alloc[s], replace=False))
    return np.concatenate(sample_idx), alloc

def chi_square_stat(observed_counts, expected_props):
    """皮尔逊卡方统计量：观测计数与「按期望比例应有的计数」的偏离程度。"""
    n = observed_counts.sum()
    expected_counts = expected_props * n
    return float(np.sum((observed_counts - expected_counts) ** 2 / expected_counts))

def monte_carlo_pvalue(stat_obs, expected_props, n_total, rng, n_sim=2000):
    """用多项分布蒙特卡洛模拟「真代表总体的样本」应有的卡方统计量分布，
    p 值 = 模拟统计量 >= 观测统计量 的比例；p 值小说明观测样本偏离真实比例太远，不像是随机抽样的结果。"""
    sims = rng.multinomial(n_total, expected_props, size=n_sim)
    sim_stats = np.array([chi_square_stat(row, expected_props) for row in sims])
    return float(np.mean(sim_stats >= stat_obs))

rng_s = np.random.default_rng(11)
pop_props = np.array([0.50, 0.25, 0.15, 0.07, 0.03])              # 5 个场景桶的真实占比
strata_values = list(range(5))
N = 20_000
population = np.repeat(np.arange(5), (pop_props * N).astype(int))
rng_s.shuffle(population)

sample_idx, alloc = stratified_sample(population, strata_values, 1000, rng_s)
sample = population[sample_idx]
sample_props = np.array([np.mean(sample == s) for s in strata_values])
print('分层抽样占比:', np.round(sample_props, 3), '| 总体占比:', pop_props)
assert np.allclose(sample_props, pop_props, atol=0.02)

obs_counts = np.array([np.sum(sample == s) for s in strata_values])
stat = chi_square_stat(obs_counts, pop_props)
p_good = monte_carlo_pvalue(stat, pop_props, 1000, rng_s, n_sim=1000)
print(f'分层抽样样本：卡方统计量 {stat:.2f}，蒙特卡洛 p 值 {p_good:.3f}')
assert p_good > 0.05, '按真实比例分层抽出来的样本不应该被判定为不具代表性'
print('\n✅ 分层抽样通过代表性检验：p 值远大于 0.05，没有理由怀疑它偏离总体分布。')

In [ ]:
# ── 对照：一份「图省事」的便利抽样（只抄了场景 0/1 前面若干条），代表性检验应该报警 ──
convenience_idx = np.concatenate([
    np.where(population == 0)[0][:800],
    np.where(population == 1)[0][:200],
])
convenience_sample = population[convenience_idx]
bad_counts = np.array([np.sum(convenience_sample == s) for s in strata_values])
bad_stat = chi_square_stat(bad_counts, pop_props)
p_bad = monte_carlo_pvalue(bad_stat, pop_props, 1000, rng_s, n_sim=1000)
print(f'便利抽样：卡方统计量 {bad_stat:.2f}，蒙特卡洛 p 值 {p_bad:.3f}')
assert p_bad < 0.01, '完全没有场景 2/3/4 的便利抽样理应被判定为不具代表性'
print('\n✅ 便利抽样被正确拦下——这正是「验证集必须做代表性检验」的意义：')
print('   光看抽样量够不够大是不够的，分布对不对也要专门验证。')

## 5 · 存储与吞吐量级计算器

车队原始数据的量级，和训练侧需要的读取吞吐量级，是数据系统设计里最容易被追问「大概多少」的两个数字。

In [ ]:
def daily_raw_volume_gb(n_vehicle, hours_active, n_cam, fps, avg_frame_kb):
    """车队每日原始数据量（GB）：车数 x 有效行驶小时 x 摄像头数 x 帧率 x 每帧大小。"""
    bytes_per_day = n_vehicle * hours_active * n_cam * fps * 3600 * avg_frame_kb * 1024
    return bytes_per_day / (1024 ** 3)

def required_throughput_mb_s(n_accelerator, imgs_per_s_per_accel, avg_img_kb):
    """训练侧所需读取吞吐（MB/s）：加速卡数 x 单卡每秒吃图数 x 单图大小。"""
    return n_accelerator * imgs_per_s_per_accel * avg_img_kb / 1024

daily_gb = daily_raw_volume_gb(n_vehicle=200, hours_active=8, n_cam=7, fps=30, avg_frame_kb=300)
print(f'200 辆车、7 路摄像头、30fps、每天 8 小时：原始数据量 ≈ {daily_gb:,.0f} GB/天 ≈ {daily_gb/1024:.0f} TB/天')
assert 1e5 < daily_gb < 1e6, '这应该是十万到百万 GB 量级（百TB到PB级），否则假设有问题'

train_mb_s = required_throughput_mb_s(n_accelerator=8, imgs_per_s_per_accel=150, avg_img_kb=300)
print(f'8 卡训练节点、单卡 150 图/秒、每图 300KB：所需读取吞吐 ≈ {train_mb_s:.0f} MB/s')
assert 100 < train_mb_s < 1000, '训练读取吞吐通常是几百 MB/s 量级'

MEDIA = [('HDD 7200rpm', 175), ('SATA SSD', 500), ('NVMe SSD', 3500), ('10GbE 网络存储', 1250)]
print(f"\n{'介质':<16} {'吞吐(MB/s)':>10}   能否撑住 {train_mb_s:.0f} MB/s 的训练读取需求")
for name, bw in MEDIA:
    ok = '✅ 够' if bw >= train_mb_s else '❌ 不够，需缓存/预取'
    print(f'{name:<16} {bw:>10}   {ok}')

print(f'\n结论：原始数据是 <strong>百 TB/天</strong> 量级，必须靠触发式回传（见 C58-03）而非全量上传；')
print(f'      训练读取吞吐是<strong>几百 MB/s</strong>量级，本地 NVMe 足够，HDD 不够。')

## 6 · 冷启动策略决策树

In [ ]:
def cold_start_strategy(has_pretrained, physically_parameterizable):
    """冷启动选择树：
    有相近领域的预训练模型 -> 迁移学习微调；
    没有预训练但问题物理上可参数化（形状/颜色/材质规则明确）-> 合成数据 + 域随机化预训练；
    两者都没有 -> 先上规则/安全默认值兜底，同时启动真实数据采集与主动学习。"""
    if has_pretrained:
        return 'transfer_learning'
    if physically_parameterizable:
        return 'synthetic_pretrain'
    return 'rule_based_fallback'

CASES = [
    (True,  True,  '新增一个和已有类别形态相似的标志类别'),
    (True,  False, '进入新国家但已有跨域预训练的检测 backbone'),
    (False, True,  '电子可变限速牌识别（内容高度可参数化）'),
    (False, False, '完全没有先验的新问题'),
]
for has_pre, param, desc in CASES:
    strategy = cold_start_strategy(has_pre, param)
    print(f'{desc:<32} -> {strategy}')

assert cold_start_strategy(True, True) == 'transfer_learning'
assert cold_start_strategy(True, False) == 'transfer_learning'
assert cold_start_strategy(False, True) == 'synthetic_pretrain'
assert cold_start_strategy(False, False) == 'rule_based_fallback'
print('\n✅ 决策树就位：核心是问「哪种先验能替代真实标注数据」，而不是死记「用迁移学习」这一个答案。')

## ✏️ 练习 1：标注质检抽检比例求解器

实现 `min_spotcheck_n(defect_rate, confidence=0.95)`：给定标注缺陷率 `defect_rate`，
求最少要抽检多少帧，才能保证「至少发现一个缺陷帧」的概率 ≥ `confidence`。

提示：抽检 n 帧一个缺陷都没抓到的概率是 $(1-\text{defect\_rate})^n$，
所以「至少抓到一个」的概率是 $1-(1-\text{defect\_rate})^n \ge \text{confidence}$，反解 $n$ 并向上取整。

In [ ]:
def min_spotcheck_n(defect_rate, confidence=0.95):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert min_spotcheck_n(0.01, 0.95) == 299
assert min_spotcheck_n(0.02, 0.99) == 228
assert min_spotcheck_n(0.10, 0.95) == 29

def catch_prob(n, defect_rate):
    return 1 - (1 - defect_rate) ** n

n_needed = min_spotcheck_n(0.01, 0.95)
assert catch_prob(n_needed, 0.01) >= 0.95
assert catch_prob(n_needed - 1, 0.01) < 0.95        # n-1 帧应该还不够
print(f'缺陷率 1% 时，抽检 {n_needed} 帧才能有 95% 把握发现至少一个缺陷帧。')
print('✅ 练习 1 通过：质检抽检比例不是拍脑袋定的 5%/10%，是缺陷率和置信度的函数。')

## ✏️ 练习 2：泄漏严重度评分器

实现 `leakage_severity(track_frac, geo_frac, device_frac, time_leak, weights=(0.4,0.3,0.2,0.1))`：
返回加权严重度分数 = `weights[0]*track_frac + weights[1]*geo_frac + weights[2]*device_frac + weights[3]*float(time_leak)`。
权重次序对应「危害排序：track > 地理 > 设备 > 时间」（呼应第 3 节的四类泄漏危害程度分析）。

In [ ]:
def leakage_severity(track_frac, geo_frac, device_frac, time_leak, weights=(0.4, 0.3, 0.2, 0.1)):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
s_bad = leakage_severity(16 / 20, 8 / 8, 3 / 3, True)
s_clean = leakage_severity(0, 0, 0, False)
assert abs(s_bad - 0.92) < 1e-9, s_bad
assert s_clean == 0.0
s_only_time = leakage_severity(0, 0, 0, True)
assert abs(s_only_time - 0.1) < 1e-9
print(f'全部泄漏: {s_bad:.2f} | 全部干净: {s_clean:.2f} | 只有时间泄漏: {s_only_time:.2f}')
print('✅ 练习 2 通过：把「有没有泄漏」升级成「泄漏有多严重」，才谈得上排优先级修复。')

## ✏️ 练习 3：学习曲线拟合反推样本量

实现 `fit_learning_curve(n1, p1, n2, p2)`：给定两个试点观测点 $(n_1,p_1),(n_2,p_2)$，
拟合 $p(n) = 1 - a n^{-b}$ 里的 $a, b$，返回 `(a, b)`。

推导：$\dfrac{1-p_1}{1-p_2} = \left(\dfrac{n_2}{n_1}\right)^{b}$，两边取对数解出 $b$，再代回任意一点解出 $a$。

In [ ]:
def fit_learning_curve(n1, p1, n2, p2):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
a_fit, b_fit = fit_learning_curve(1000, 0.6, 4000, 0.75)
assert abs(learning_curve(1000, a_fit, b_fit) - 0.6) < 1e-9
assert abs(learning_curve(4000, a_fit, b_fit) - 0.75) < 1e-9

n_target = required_n_closed_form(0.9, a_fit, b_fit)
assert n_target > 4000, '目标指标 0.9 高于试点 2 的 0.75，需要的样本量必须更多'
print(f'拟合得到 a={a_fit:.3f}, b={b_fit:.3f}；要把指标从 0.75（n=4000）推到 0.9，需要约 {n_target:,.0f} 个样本。')
print('✅ 练习 3 通过：两个试点观测点，就能把「还要标多少数据」从直觉变成一个可算的数。')

## ✏️ 练习 4：分层抽样的预算保底分配器

实现 `stratified_alloc_with_floor(strata_counts, n_total, min_per_stratum)`：
按各层在总体中的占比比例分配抽样名额（向下取整），但每层至少分到 `min_per_stratum` 个名额——
即便这会让总分配量超过 `n_total`（这正是「保证稀有层被看见」要付出的代价）。

In [ ]:
def stratified_alloc_with_floor(strata_counts, n_total, min_per_stratum):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
strata_counts = {'A': 10000, 'B': 5000, 'C': 500, 'D': 50}
alloc = stratified_alloc_with_floor(strata_counts, 1000, min_per_stratum=30)
assert alloc == {'A': 643, 'B': 321, 'C': 32, 'D': 30}
assert sum(alloc.values()) == 1026          # 超过 n_total=1000，因为 D 层被保底拉到了 30
print('分配结果：', alloc, '| 总分配量：', sum(alloc.values()), '（原始预算 1000）')
print('✅ 练习 4 通过：稀有层 D 本该按比例只分到 3 个名额，保底机制把它拉到 30，')
print('   代价是总分配量超出预算 26 个——这个超支就是「不让稀有层归零」需要付出的真实代价。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def min_spotcheck_n(defect_rate, confidence=0.95):
    if defect_rate <= 0:
        return float('inf')
    n = math.log(1 - confidence) / math.log(1 - defect_rate)
    return math.ceil(n)

In [ ]:
# 练习 2 参考答案
def leakage_severity(track_frac, geo_frac, device_frac, time_leak, weights=(0.4, 0.3, 0.2, 0.1)):
    return (weights[0] * track_frac + weights[1] * geo_frac
            + weights[2] * device_frac + weights[3] * float(time_leak))

In [ ]:
# 练习 3 参考答案
def fit_learning_curve(n1, p1, n2, p2):
    b = math.log((1 - p1) / (1 - p2)) / math.log(n2 / n1)
    a = (1 - p1) * (n1 ** b)
    return a, b

In [ ]:
# 练习 4 参考答案
def stratified_alloc_with_floor(strata_counts, n_total, min_per_stratum):
    total = sum(strata_counts.values())
    alloc = {}
    for s, c in strata_counts.items():
        raw = int(n_total * c / total)
        alloc[s] = max(min_per_stratum, raw)
    return alloc

---
## 🧪 真实工程胶囊：数据系统设计文档模板 + 面试口播要点

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 数据系统设计文档模板（面试白板可以照这个骨架展开）
# ══════════════════════════════════════════════════════════════════════
# 1. 来源画像：自采 / 公开 / 合成 / 众包 各覆盖哪一块缺口，给出成本-质量-合规的对比
# 2. 标注体系：规范（含边界案例判例）+ 质检三层防线（黄金集/多标共识/模型回环）
#              + 成本模型（基础+返工+QC）+ 外包/自建的阶段性判断
# 3. 划分方案：分组键有哪些（track_id/location_id/device_id/timestamp），
#              按哪个粒度分组切分，是否做过四类泄漏检测
# 4. 长尾与偏差：类别频次分布 + 场景覆盖率检查（对照目标 ODD 的先验占比）
# 5. 隐私合规：脱敏对象、脱敏发生在哪一层、脱敏模型自身的召回率指标、留存期限
# 6. 版本与血缘：数据集快照怎么标识、训练如何引用具体版本、能否反查血缘
# 7. 冷启动：新任务/新地区上线时走哪条路径，安全兜底是什么
# 8. 存储与吞吐：车队原始数据量级、触发回传后的实际上传量级、训练读取吞吐需求

# ══════════════════════════════════════════════════════════════════════
# B. 面试里最容易被追问的三句话，提前想好怎么答
# ══════════════════════════════════════════════════════════════════════
# Q: "你怎么保证训练集和测试集没有泄漏？"
# A: "我会按 track_id / location_id / device_id 做分组切分而不是随机切分，
#     并且用滚动时间窗口保证训练集严格早于测试集；上线前我会跑一次四类泄漏检测。"
#
# Q: "标注质量怎么保证？"
# A: "三层质检：黄金集抽检看标注员理解是否到位，多标共识量化标注者间一致性，
#     模型回环质检零成本地抓系统性错误——三层组合而不是只做一层。"
#
# Q: "数据不够怎么办？"
# A: "先看这是长尾问题还是冷启动问题：长尾用重采样/重加权/copy-paste 处理（见 C58），
#     全新任务用冷启动选择树——有预训练用迁移学习，问题可参数化用合成数据，否则先上
#     规则兜底同时启动主动学习。"

# ══════════════════════════════════════════════════════════════════════
# C. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 长尾处理的具体算法（重采样/重加权/logit adjustment）  -> C58 模块 01（本课只讲识别）
# · 数据闭环的触发/挖掘/基础设施细节                        -> C58 模块 03-05（本课只讲面试组织方式）
# · 数据工程的实现细节（存储格式/流水线代码）                -> C43（本课只讲设计取舍）
# · TSR 数据集全景与标志分类体系                            -> C55 模块 01（本课直接引用其结论）
# · 通用 Fermi 估算方法                                    -> C65 模块 01（本课只给数据系统专属锚点数字）
'''
print(RECIPE)
for token in ['track_id', 'location_id', '滚动时间窗口', '三层质检', '冷启动选择树', 'C58 模块 01', 'C55 模块 01']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：设计文档骨架 / 高频追问的标准答法 / 与其他课程的分工边界')

### 小结

- **数据来源没有万能解**：自采/公开/合成/众包在成本-质量-合规三个维度上几乎互补，
  面试里最加分的回答是「用哪种来源覆盖哪一段缺口」，而不是选一种。
- **标注体系要能回答「标签能不能被信任」**：规范里的边界案例判例、三层质检防线
  （黄金集/多标共识/模型回环）、以及把成本拆成「基础+返工+QC」三项，都是可以当场讲清楚的具体机制。
- **随机划分在感知数据里几乎总是错的**：track / 地理 / 设备 / 时间四类泄漏会让离线指标被系统性高估，
  正确做法是按分组键做 group-based split，时间维度用滚动窗口而不是全局随机。
- **长尾和偏差是两个不同的问题**：长尾问的是「某类别是不是太少」，偏差问的是「采集分布和部署分布是否匹配」——
  一个类别样本多但都来自同一路口，照样是坏数据。
- **合规是数据能不能合法存在的前提，不是附加项**：脱敏模型本身也有召回率，需要像业务模型一样被单独评测。
- **冷启动的本质是问「哪种先验能替代真实标注数据」**：预训练模型给视觉特征先验，规则系统给领域知识先验，
  合成数据给物理生成过程先验——先验越强，需要的真实数据越少。
- **数量级心算是免费的说服力**：车队原始数据是百 TB/天量级，训练读取吞吐是几百 MB/s 量级，
  两个数字能瞬间判断「要不要为了这个过度设计分布式存储」。

下一站：**模块 03 · 建模与评测设计** —— 数据系统就位之后，
下一个最容易在面试里丢分的地方，是跳过 baseline 直接谈架构新颖度。